<a href="https://colab.research.google.com/github/MBR4V0/Iniciacao-Cientifica/blob/main/MongoDB_CONECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pymongo dnspython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 11.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pymongo]


In [8]:
!pip install -q pymongo dnspython pandas odfpy openpyxl

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
from google.colab import userdata
from pymongo import MongoClient

# Recupera a URL protegida
MONGO_URI = userdata.get('MONGO_URI')

# Cria a conexão
client = MongoClient(
    MONGO_URI,
    serverSelectionTimeoutMS=10000
)

# Testa a conexão
try:
    client.admin.command('ping')
    print(" CONECTADO AO MONGODB COM SUCESSO!")
except Exception as e:
    print(" ERRO NA CONEXÃO:", e)

 CONECTADO AO MONGODB COM SUCESSO!


In [11]:

import pandas as pd
from pymongo import MongoClient
from google.colab import files, userdata
import io
import os

# ====================================
# 1. CONECTAR AO MONGODB
# ====================================

MONGO_URI = userdata.get("MONGO_URI")

client = MongoClient(
    MONGO_URI,
    serverSelectionTimeoutMS=15000
)

client.admin.command("ping")

print("✅ MongoDB conectado!")

# ====================================
# 2. DEFINIR BANCO E COLEÇÃO
# ====================================

db = client["alunos"]
collection = db["importacao_csv"]

# ====================================
# 3. SELECIONAR ARQUIVO
# ====================================

print("Selecione sua planilha:")
uploaded = files.upload()

nome_arquivo = list(uploaded.keys())[0]

extensao = os.path.splitext(nome_arquivo)[1].lower()

# ====================================
# 4. LER AUTOMATICAMENTE
# ====================================

arquivo = io.BytesIO(uploaded[nome_arquivo])

if extensao == ".ods":
    df = pd.read_excel(arquivo, engine="odf")

elif extensao in [".xlsx", ".xls"]:
    df = pd.read_excel(arquivo)

elif extensao == ".csv":
    try:
        df = pd.read_csv(
            arquivo,
            sep=None,
            engine="python",
            encoding="utf-8"
        )
    except UnicodeDecodeError:
        arquivo.seek(0)
        df = pd.read_csv(
            arquivo,
            sep=None,
            engine="python",
            encoding="latin1"
        )

else:
    raise ValueError("Formato não suportado: " + extensao)

print("\n✅ Planilha carregada!")
print("Arquivo:", nome_arquivo)
print("Registros:", len(df))
print("Colunas:", list(df.columns))

display(df.head())

# ====================================
# 5. IMPORTAR PARA O MONGODB
# ====================================

dados = df.astype(object).where(
    pd.notna(df), None
).to_dict(orient="records")

if dados:
    resultado = collection.insert_many(dados)

    print("\n🎉 IMPORTAÇÃO REALIZADA!")
    print("Registros enviados:", len(resultado.inserted_ids))
    print("Banco:", db.name)
    print("Coleção:", collection.name)

else:
    print("A planilha está vazia.")

# ====================================
# 6. CONFERIR IMPORTAÇÃO
# ====================================

print(
    "\nTotal de documentos na coleção:",
    collection.count_documents({})
)

client.close()

✅ MongoDB conectado!
Selecione sua planilha:


Saving BRCA.datafreeze.20120227.xlsx to BRCA.datafreeze.20120227.xlsx

✅ Planilha carregada!
Arquivo: BRCA.datafreeze.20120227.xlsx
Registros: 466
Colunas: ['Sample', 'bcr_patient_barcode', 'Tissue', 'PAM50', 'Triple Negative', 'Batch', 'Histology', 'UNC.microarray', 'BCGSC.miRNAseqs', 'Broad.CN', 'Methylation', 'WU.seq.20111103', 'RPPA', 'Integration.Freeze', 'Integration.RPPA.Freeze.batch1']


/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,Sample,bcr_patient_barcode,Tissue,PAM50,Triple Negative,Batch,Histology,UNC.microarray,BCGSC.miRNAseqs,Broad.CN,Methylation,WU.seq.20111103,RPPA,Integration.Freeze,Integration.RPPA.Freeze.batch1
0,TCGA-A1-A0SK-01A,TCGA-A1-A0SK,Tumor,Basal,TN,72,Breast - Other,1,1,1,1,1,1,Freeze.466,Freeze.348
1,TCGA-A2-A0ST-01A,TCGA-A2-A0ST,Tumor,Basal,TN,72,Infiltrating Ductal Carcinoma,1,1,1,1,1,1,Freeze.466,Freeze.348
2,TCGA-A2-A0SX-01A,TCGA-A2-A0SX,Tumor,Basal,TN,72,Infiltrating Ductal Carcinoma,1,1,1,1,1,1,Freeze.466,Freeze.348
3,TCGA-A2-A0T2-01A,TCGA-A2-A0T2,Tumor,Basal,TN,72,Infiltrating Ductal Carcinoma,1,1,1,1,1,1,Freeze.466,Freeze.348
4,TCGA-BH-A0E6-01A,TCGA-BH-A0E6,Tumor,Basal,TN,56,Infiltrating Ductal Carcinoma,1,1,1,1,1,1,Freeze.466,Freeze.348



🎉 IMPORTAÇÃO REALIZADA!
Registros enviados: 466
Banco: alunos
Coleção: importacao_csv

Total de documentos na coleção: 466
